# Decoding

### Greedy Decoding 

- 언어모델이 텍스트를 한 토큰씩 만들어내는 과정을 눈으로 보여주는 코드 

- 예) "Transformers are the" 라는 문장 다음에 뭐가 이어질지 n 단계에 걸쳐 예측

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [9]:
# GREEDY : 제일 확률 높음

input_txt = "Transformers are the"

encoded = tokenizer(
    input_txt,
    return_tensors="pt",
)

input_ids = encoded["input_ids"].to(device)

iterations = []

n_steps = 8
choice_per_step = 5

with torch.inference_mode():
    for _ in range(n_steps):
        iteration = {
            "input": tokenizer.decode(
                input_ids[0],
                skip_special_tokens=True,
            )
        }

        outputs = model(input_ids=input_ids)

        # shape: [vocab_size]
        next_token_logits = outputs.logits[0, -1, :]
        next_token_probs = torch.softmax(next_token_logits, dim=-1)

        # 확률이 높은 토큰 5개
        top_probs, top_ids = torch.topk(
            next_token_probs,
            k=choice_per_step,
        )

        for choice_idx, (token_prob, token_id) in enumerate(
            zip(top_probs, top_ids),
            start=1,
        ):
            token_text = tokenizer.decode([token_id.item()])

            iteration[f"Choice {choice_idx}"] = (
                f"{token_text!r} ({token_prob.item() * 100:.2f}%)"
            )

        # Greedy decoding : 확률이 가장 높은 토큰 선택
        next_token_id = top_ids[0].reshape(1, 1)

        input_ids = torch.cat(
            [input_ids, next_token_id],
            dim=-1,
        )

        iterations.append(iteration)

greedy_result = pd.DataFrame(iterations)
greedy_output = tokenizer.decode(input_ids[0], skip_special_tokens=True)
print(greedy_output)
greedy_result

Transformers are the most common type of particle. They are


,input,Choice 1,Choice 2,Choice 3,Choice 4,Choice 5
0,Transformers are the,' most' (9.76%),' same' (2.94%),' only' (2.87%),' best' (2.38%),' first' (1.77%)
1,Transformers are the most,' common' (22.90%),' powerful' (6.88%),' important' (6.32%),' popular' (3.95%),' commonly' (2.14%)
2,Transformers are the most common,' type' (15.06%),' types' (3.31%),' form' (1.91%),' way' (1.89%),' and' (1.49%)
3,Transformers are the most common type,' of' (83.13%),' in' (3.16%),'.' (1.92%),"',' (1.63%)",' for' (0.88%)
4,Transformers are the most common type of,' particle' (1.55%),' object' (1.02%),' light' (0.71%),' energy' (0.67%),' objects' (0.66%)
5,Transformers are the most common type of particle,'.' (14.26%),' in' (11.57%),' that' (10.19%),"',' (9.57%)",' accelerator' (5.81%)
6,Transformers are the most common type of parti...,' They' (17.48%),'\n' (15.19%),' The' (7.06%),' These' (3.09%),' In' (3.07%)
7,Transformers are the most common type of parti...,' are' (38.78%),' have' (8.14%),' can' (7.98%),"""'re"" (5.04%)",' consist' (1.57%)


### Beam Search 

- 매 스텝마다 상위 후보를 1개가 아니라 k개(beam width) 유지하면서, 여러 가능성 있는 문장 후보를 동시에 살펴봄

- Grddy Decoding에 비해 탐색 범위가 넓고, 계산 비용이 k배 정도 높다. 
  결과 품질은 더 일관성 있고 자연스러운 경향이 있으며 둘 다 최적해 보장은 할 수 없다.

In [11]:
input_txt = "Transformers are the"
encoded = tokenizer(
    input_txt,
    return_tensors="pt",
)
input_ids = encoded["input_ids"].to(device)

# --- 빔서치 설정 ---
n_steps = 8
beam_width = 5

# --- 초기 beam: 시작 시퀀스 1개, log_prob 0 ---
beams = [
    {
        "token_ids": input_ids,
        "log_prob": 0.0,
    }
]

iterations = []

with torch.inference_mode():
    for step in range(n_steps):
        candidates = []  # 이번 스텝에서 나올 모든 후보 초기화

        # 현재 살아 있는 각 beam을 확장
        for beam in beams:
            current_ids = beam["token_ids"]
            current_log_prob = beam["log_prob"]

            outputs = model(input_ids=current_ids)

            # 현재 문장의 마지막 위치에서 다음 토큰 예측
            next_token_logits = outputs.logits[0, -1, :]

            next_token_log_probs = torch.log_softmax(
                next_token_logits,
                dim=-1,
            )

            # 현재 beam에서 확률이 높은 토큰 beam_width개
            top_log_probs, top_ids = torch.topk(
                next_token_log_probs,
                k=beam_width,
            )

            # 선택된 토큰마다 새로운 문장 후보 생성
            for token_log_prob, token_id in zip(
                top_log_probs,
                top_ids,
            ):
                next_token_id = token_id.reshape(1, 1)

                new_ids = torch.cat(
                    [current_ids, next_token_id],
                    dim=-1,
                )

                # 로그 확률은 곱셈 대신 덧셈
                new_log_prob = current_log_prob + token_log_prob.item()

                candidates.append(
                    {
                        "token_ids": new_ids,
                        "log_prob": new_log_prob,
                    }
                )

        # 모든 후보를 누적 로그 확률 기준으로 정렬
        candidates = sorted(
            candidates,
            key=lambda candidate: candidate["log_prob"],
            reverse=True,
        )

        # 상위 beam_width개 문장만 유지
        beams = candidates[:beam_width]

        # 현재 단계의 beam들을 DataFrame에 기록
        iteration = {
            "step": step + 1,
        }

        for beam_idx, beam in enumerate(beams, start=1):
            beam_text = tokenizer.decode(
                beam["token_ids"][0],
                skip_special_tokens=True,
            )

            iteration[f"Beam {beam_idx}"] = (
                f"{beam_text!r} (logp={beam['log_prob']:.3f})"
            )

        iterations.append(iteration)

beam_result = pd.DataFrame(iterations)
best_beam = beams[0]
beam_output = tokenizer.decode(best_beam["token_ids"][0], skip_special_tokens=True)
print(beam_output)
beam_result

Transformers are the most important part of the game, and


,step,Beam 1,Beam 2,Beam 3,Beam 4,Beam 5
0,1,'Transformers are the most' (logp=-2.327),'Transformers are the same' (logp=-3.528),'Transformers are the only' (logp=-3.551),'Transformers are the best' (logp=-3.738),'Transformers are the first' (logp=-4.032)
1,2,'Transformers are the most common' (logp=-3.801),'Transformers are the same as' (logp=-4.558),'Transformers are the most powerful' (logp=-5....,'Transformers are the most important' (logp=-5...,'Transformers are the most popular' (logp=-5.559)
2,3,'Transformers are the most common type' (logp=...,'Transformers are the same as the' (logp=-6.560),'Transformers are the same as in' (logp=-6.874),'Transformers are the most common types' (logp...,'Transformers are the most important part' (lo...
3,4,'Transformers are the most common type of' (lo...,'Transformers are the most important part of' ...,'Transformers are the most common types of' (l...,'Transformers are the same as in the' (logp=-8...,'Transformers are the same as the ones' (logp=...
4,5,'Transformers are the most important part of t...,'Transformers are the same as in the original'...,'Transformers are the most important part of a...,'Transformers are the most common type of part...,'Transformers are the most important part of y...
5,6,'Transformers are the most important part of t...,'Transformers are the same as in the original ...,'Transformers are the most common type of part...,'Transformers are the most common type of part...,'Transformers are the most important part of t...
6,7,'Transformers are the most important part of t...,'Transformers are the most important part of t...,'Transformers are the same as in the original ...,'Transformers are the same as in the original ...,'Transformers are the most important part of t...
7,8,'Transformers are the most important part of t...,'Transformers are the most important part of t...,'Transformers are the same as in the original ...,'Transformers are the same as in the original ...,'Transformers are the most important part of t...


### Top-k

- 확률에 따라 무작위로 뽑는 방법
- 같은 입력 -> 같은 결과가 위의 두 방법은 동일하지만, Top-k 샘플링은 매번 다를 수 있다. (stochastic)
- 다양하고 자연스러운 문장을 만드는 방법

In [12]:
input_txt = "Transformers are the"
encoded = tokenizer(
    input_txt,
    return_tensors="pt",
)
input_ids = encoded["input_ids"].to(device)

# --- top-k 샘플링 설정 ---
n_steps = 8
top_k = 5

iterations = []

with torch.inference_mode():
    for _ in range(n_steps):
        iteration = {
            "input": tokenizer.decode(
                input_ids[0],
                skip_special_tokens=True,
            )
        }

        outputs = model(input_ids=input_ids)
        # 현재 문장의 마지막 위치에서 다음 토큰 예측
        next_token_logits = outputs.logits[0, -1, :]

        # 확률이 높은 토큰 top_k개만 선택
        top_logits, top_ids = torch.topk(
            next_token_logits,
            k=top_k,
        )

        # top_k 후보 안에서만 확률 계산
        top_probs = torch.softmax(
            top_logits,
            dim=-1,
        )

        # 후보 확인 (기록용)
        for choice_idx, (token_prob, token_id) in enumerate(
            zip(top_probs, top_ids),
            start=1,
        ):
            token_text = tokenizer.decode([token_id.item()])
            iteration[f"Choice {choice_idx}"] = (
                f"{token_text!r} ({token_prob.item() * 100:.2f}%)"
            )

        # top_k 후보의 확률분포에서 1개 샘플링
        sampled_index = torch.multinomial(
            top_probs,
            num_samples=1,
        )

        # 실제로 선택된 토큰 id 뽑기
        next_token_id = top_ids[sampled_index].reshape(1, 1)

        input_ids = torch.cat(
            [input_ids, next_token_id],
            dim=-1,
        )

        iterations.append(iteration)

topk_result = pd.DataFrame(iterations)
topk_output = tokenizer.decode(input_ids[0], skip_special_tokens=True)
print(topk_output)
topk_result

Transformers are the only type of data that can be used


,input,Choice 1,Choice 2,Choice 3,Choice 4,Choice 5
0,Transformers are the,' most' (49.49%),' same' (14.89%),' only' (14.56%),' best' (12.07%),' first' (9.00%)
1,Transformers are the only,' ones' (45.40%),' way' (19.74%),' two' (16.16%),' type' (10.48%),' non' (8.22%)
2,Transformers are the only type,' of' (83.71%),' that' (10.38%),' to' (2.18%),' in' (2.17%),' which' (1.55%)
3,Transformers are the only type of,' object' (33.36%),' data' (19.06%),' particle' (16.62%),' energy' (15.65%),' objects' (15.32%)
4,Transformers are the only type of data,' that' (66.15%),' to' (14.60%),' in' (6.90%),' which' (6.90%),' we' (5.45%)
5,Transformers are the only type of data that,' can' (62.20%),' is' (12.97%),' are' (11.41%),' we' (8.36%),' you' (5.06%)
6,Transformers are the only type of data that can,' be' (92.13%),"""'t"" (2.80%)",' contain' (1.93%),' have' (1.92%),' store' (1.22%)
7,Transformers are the only type of data that ca...,' used' (29.06%),' stored' (27.75%),' passed' (16.26%),' converted' (14.72%),' accessed' (12.21%)


In [15]:
inputs = tokenizer(
    "The future of AI is",
    return_tensors="pt",
)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    top_k=50,
    top_p=0.9,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id,  # 이거 추가해주면 경고 사라짐
)

In [16]:
inputs = tokenizer("The future of AI is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    num_beams=5,       # Beam Search
    do_sample=False,   # Sampling 비활성화
)